In [1]:
print("hi")

hi


In [2]:
# Step1: Create & Setup hugging face API token in Collab

In [3]:
# Step2: install the dependencies ]

In [4]:
!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git # Also get the latest version Unsloth!

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [5]:
# Step3: Import necessary libraries

from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
# Step4: Check HF token

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [7]:
# Optional: Check GPU availability
# Test if CUDA is available
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU device: Tesla T4


In [8]:

# Step5: Setup pretrained DeepSeek-R1

model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 2048
dtype = None
load_in_4bit = True

import os
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"  #did this to avoide time out error, This switches to ModelScope mirror, which often works faster.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_sequence_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token
)

==((====))==  Unsloth 2026.3.7: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [9]:

# Step6: Setup system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Before crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}

### Answer:
{}
"""

In [ ]:
# Step7: Run Inference on the model

# Define a test question
question = """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
              sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
              what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

FastLanguageModel.for_inference(model)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

print(inputs.input_ids.shape) #to debug

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = False,                     #use_cache=False avoids Unsloth's RoPE cache shape mismatch bug, (shape [1,32,1,128] vs [1,32,N,128] during prefill)
    pad_token_id=tokenizer.eos_token_id    #pad_token_id must be set to avoid undefined padding behaviour
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)  #skip_special_tokens=True gives cleaner output


print(response)
# print(response[0].split("### Answer:")[1])



In [21]:
# Step8: Setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split = "train[:500]")

In [12]:
# Step9: Setup/Apply LoRA finetuning to the model
# W = 4096 × 4096
# A = 4096 × 16, B = 16 × 4096 (bcz r = 16)  // this is LoRA setup step

model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3047,
    use_rslora = False,
    loftq_config = None
)

Unsloth 2026.3.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [13]:
# Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
    del model._unwrapped_old_generate

In [24]:
# Normal training:
# 4096 × 4096             //i.e. W = 4096 × 4096
# = 16,777,216 parameters

# LoRA training:
# 4096 × 16 + 16 × 4096   //i.e. [ΔW] = [A] x [B] = 4096 × 16 + 16 × 4096,
# = 131,072 parameters    // Where ΔW is learned during training.

# Define a formatting function for the dataset
def formatting_prompts_func(examples):
    # Debugging: Print column names to check available keys
    # print("Dataset column names:", examples.keys())
    instructions = examples["Question"]
    outputs = examples["Response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Apply the prompt style and add EOS token
        text = prompt_style.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return texts

print("Debugging: Dataset column names:", medical_dataset.column_names)

trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = medical_dataset,
    # Remove dataset_text_field as formatting_func will provide the text
    formatting_func = formatting_prompts_func,
    max_seq_length = max_sequence_length,
    dataset_num_proc = 1,

    # Define training args
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir = "outputs",
    ),
)

Debugging: Dataset column names: ['Question', 'Complex_CoT', 'Response']


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/500 [00:00<?, ? examples/s]

In [26]:
# Setup WANDB
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aglawechakuli (aglawechakuli-bia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [32]:
# Start the fine-tuning process
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.152250
20,0.984115
30,1.035674
40,1.057705
50,1.085482
60,1.042882


In [34]:
wandb.finish()

In [35]:
# Step10: Testing after fine-tuning
question = """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing
              but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
              what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = False,                     #use_cache=False avoids Unsloth's RoPE cache shape mismatch bug, (shape [1,32,1,128] vs [1,32,N,128] during prefill)
    pad_token_id=tokenizer.eos_token_id    #pad_token_id must be set to avoid undefined padding behaviour
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)  #skip_special_tokens=True gives cleaner output

print(response[0])

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response.split("### Answer:")[1].strip())

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Belowisataskdescriptionalongwithadditionalcontextprovidedintheinputsection.Yourgoalistoprovideawell-reasonedresponsethateffectivelyaddressestherequest.Beforecraftingyouranswer,takeamomenttocarefullyanalyzethequestion.Developaclear,step-by-stepthoughtprocesstoensureyourresponseisbothlogicalandaccurate.###Task:Youareamedicalexpertspecializinginclinicalreasoning,diagnostics,andtreatmentplanning.Answerthemedicalquestionbelowusingyouradvancedknowledge.###Query:A61-year-oldwomanwithalonghistoryofinvoluntaryurinelossduringactivitieslikecoughingorsneezingbutnoleakageatnightundergoesagynecologicalexamandQ-tiptest.Basedonthesefindings,whatwouldcystometrymostlikelyrevealaboutherresidualvolumeanddetrusorcontractions?###Answer:Basedonthesefindings,cystometrywouldmostlikelyrevealthatthepatient'sresidualurinaryvolumeisapproximately300-500mL,anddetrusorcontractionswouldlikelyexceed100mL.Thesefindingsarecharacteristicofstressurinaryincontinence,whereinvoluntaryurinelossoccursduringactivitieslikescoughi

In [40]:
# test 2
question = """A 59-year-old man presents with a fever, chills, night sweats, and generalized fatigue,
              and is found to have a 12 mm vegetation on the aortic valve. Blood cultures indicate gram-positive, catalase-negative,
              gamma-hemolytic cocci in chains that do not grow in a 6.5% NaCl medium.
              What is the most likely predisposing factor for this patient's condition?"""

FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = False
)

# Decode the response tokens back to text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "### Answer:" in response:
    print(response.split("### Answer:")[1].strip())
else:
    print(response)

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Belowisataskdescriptionalongwithadditionalcontextprovidedintheinputsection.Yourgoalistoprovideawell-reasonedresponsethateffectivelyaddressestherequest.Beforecraftingyouranswer,takeamomenttocarefullyanalyzethequestion.Developaclear,step-by-stepthoughtprocesstoensureyourresponseisbothlogicalandaccurate.###Task:Youareamedicalexpertspecializinginclinicalreasoning,diagnostics,andtreatmentplanning.Answerthemedicalquestionbelowusingyouradvancedknowledge.###Query:A59-year-oldmanpresentswithafever,chills,nightsweats,andgeneralizedfatigue,andisfoundtohavea12mmvegetationontheaorticvalve.Bloodculturesindicategram-positive,catalase-negative,gamma-hemolyticcocciinchainsthatdonotgrowina6.5%NaClmedium.Whatisthemostlikelypredisposingfactorforthispatient'scondition?###Answer:Thesymptomsandtestresultsdescribedalignwithendocarditisduetoacarbobacterpneumoniae,whichiscommonlyassociatedwithaorticvalvevegetations.Thefactthatthecocciaregram-positive,gamma-hemolytic,anddonotgrowina6.5%NaClmediumsuggeststhatthey

🚀 Complete Training Architecture --

Dataset

   ↓

Prompt Formatting

   ↓

Tokenizer

   ↓

4-bit Quantized Model

   ↓

LoRA Adapters

   ↓

Forward Pass

   ↓

Cross Entropy Loss

   ↓

Backpropagation

   ↓

Update LoRA weights only